#  プログラミング言語 AIチュータ
#  Programming Languages Hands-on Environment

In [ ]:
! pip install openai
! jupyter labextension install @jupyter-widgets/jupyterlab-manager

In [ ]:
import sqlite3
import json
import logging
import time
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, Markdown
from IPython.core.magic import register_line_magic, register_cell_magic, register_line_cell_magic
import openai

DBG=2

topic_menu = [
    "Numbers and Arithmetic Expressions ",
    "Simple Printing",
    "Defining Functions",
    "Conditional Expressions/Statements",
    "Variable Definitions",
    "Iterations (while, for, etc.)",
    ("Builtin Types",
     ["Strings",
      "Lists",
      "Arrays"]),
    ("Defining new types",
     ["Struct",
      "Variant",
      "Class"]),
    "Defining Recursive Types",
    "Recursive Functions",
    "Solving Problems with Recursions",
    ("Dealing with Errors",
     ["Exceptions",
      "Using the result type"]),
    "Modules and Standard Library",
    "Other (describe below)"
]

python_system_prompt = """
あなたはPythonプログラミングの陽気な優しいチュータです。
Pythonプログラミングのあるトピックについて、
- コーディングの問題を出してと言われたらコードを書かせる問題を出してください。
- リーディングの問題を出してと言われたら、コードを読ませてどういう入力に対して何を出力するプログラムか答えさせる問題を出してください。
- デバッグの問題を出してと言われたら、仕様とそれに対する少し間違ったコードを与えてどこが間違っているかを答えさせる問題を出してください。
- 解説をしてと言われたら、そのトピックの一般的な解説をしてください。問題を出す必要はないです。
間違った答えや、ヒントをくださいの求めには、答えは教えずにヒントを出してあげてください。
降参です、と言われたら答えを教えてあげて、似た問題をもう一問出しましょうかと言って、それを解くよう促してください。

トピックは以下の順序で教えることとします

算術式、数値の表現
関数定義
条件分岐
変数
繰り返し
再帰関数定義
再帰関数呼び出しを使った問題解決
文字列
リスト、タプル
辞書
新しいデータ型の定義
再帰的なデータ型の活用
ライブラリの利用(import文)
総合問題

各トピックについて問題を出すときに, それ以降に習うトピックの知識を前提にして出題してはいけません。
それ以前のトピックについてはもう学んだと仮定して良いです。
例えば「繰り返し」についての問題を出すときは、「組み込みのデータ構造（配列、リストなど）」に関する知識を前提にしてはいけませんが、「算術式、数値の表現」や「変数」についてはもう学んだものとして良いです。
関数定義およびそれ以降のコーディングの問題では、入力を指定して、適切な返り値を返す関数を書かせる問題にしてください。
関数名も指定してください。テストコードをいくつか書いてそれが通るようにせよと指示してください。
関数定義およびそれ以降のリーディングの問題では関数を与えてそれについて答えさせてください。
関数定義およびそれ以降のデバッグの問題では関数を与えてそれの間違いを答えさせてください。
"""

generic_system_prompt = """
You are a cheerful and kind tutor for programming in Go, Julia, OCaml, and Rust. Regarding a specific topic in programming:
* When asked to give a coding problem, provide a problem where the student must write code.
* When asked to give a reading problem, provide a piece of code and ask the student to explain what it outputs for a specific input.
* When asked to give a debugging problem, provide the specification and a slightly incorrect version of the code and ask the student to identify the mistake.
* When asked to give an explanation, provide a general explanation of the topic with an example or two without giving a problem.
* When a topic does not exist in a particular programming language, suggest a topic close to it

The topics will be taught in the following order:
* Integers, Floating Point Numbers, and Arithmetic Expressions
* Simple Printing 
* Defining Functions
* Conditional Expressions/Statements (if, case, etc.)
* Variable Definitions
* Iterations (while, for, etc.)
* Builtin Types
  * Strings
  * Lists
  * Arrays
* Defining new types
  * Struct
  * Variant
  * Class
* Defining Recursive Types
* Recursive Functions
* Solving Problems with Recursions
* Dealing with Errors
  * Exceptions
  * Using the result type
* Modules and Standard Library

When giving problems on a topic, do not assume knowledge of later topics, but assume earlier topics have been learned.  For example, a problem on "Conditional Expressions/Statements" must not assume knowledge on "Iterations" but can assume knowledge on "Defining Functions"  For coding problems on "Defining Functions" and later topics, specify the inputs and ask for a function that returns the correct output. Also, specify the function name and provide test code that must pass. For reading problems on "Defining Functions" and later topics, provide a function and ask the student to explain it. For debugging problems on "Defining Functions" and later topics, provide a specification and a function including a bug and ask the student to identify the mistake.
"""

# OpenAI や urllib3 の INFO ログを無効化
logging.getLogger("openai").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)

class dialog_store:
    def __init__(self, a_sqlite, system_prompt, trunc):
        self.a_sqlite = a_sqlite
        self.uncommitted = []
        self.messages = []
        self.create_db(a_sqlite, system_prompt, trunc)
    def query(self, q, *vals):
        return self.conn.execute(q, vals)
    def create_db(self, a_sqlite, system_prompt, trunc):
        if trunc and os.path.exists(a_sqlite):
            os.remove(a_sqlite)
        self.conn = sqlite3.connect(a_sqlite)
        self.query("create table if not exists messages(ts, msg)")
        self.load()
        if len(self.messages) == 0:
            self.add({"role": "system", "content": system_prompt})
    def load(self):
        for ts, msg in self.query("select * from messages"):
            self.messages.append(json.loads(msg))
    def add_db(self, msg):
        ts = time.time()
        msg_s = json.dumps(msg, ensure_ascii=False)
        self.query("insert into messages (ts, msg) values (?, ?)", ts, msg_s)
    def commit(self):
        self.conn.commit()
    def add(self, msg):
        self.uncommitted.append(msg)
        self.messages.append(msg)
    def sync(self):
        for msg in self.uncommitted:
            self.add_db(msg)
        self.commit()
        self.uncommitted[:] = []

class tutor_client:
    def __init__(self, a_sqlite, system_prompt, trunc):
        self.client = openai.AzureOpenAI(
            azure_endpoint="https://deepseektesthu5843508078.openai.azure.com/",
            api_key=os.environ["AZURE_OPENAI_API_KEY"],
            api_version="2024-08-01-preview")
        self.dialog = dialog_store(a_sqlite, system_prompt, trunc)
        self.model = "gpt-4o"
        
    def send(self, prompt):
        """
        prompt を送る; 受け取った返事を返す
        """
        if not prompt.strip():
            return
        self.dialog.add({"role": "user", "content": prompt})
        rep = self.client.chat.completions.create(
            model=self.model,
            messages=self.dialog.messages
        )
        res = rep.choices[0].message.content
        self.dialog.add({"role": "assistant", "content": res})
        self.dialog.sync()
        return res
        
    def send_and_disp(self, prompt, res_wrapper):
        """
        prompt を送る; 受け取った返事を Markdownで, セル実行の結果として表示
        """
        res = self.send(prompt)
        display(Markdown(res_wrapper(res)))
        return res
        
    def send_and_disp_in_area(self, prompt, res_wrapper, output_area):
        """
        prompt を送る; 受け取った返事を Markdownで, 指定した output_area に表示
        """
        with output_area:
            display(Markdown(f"⏳ 生成中... "))
            self.send_and_disp(prompt, res_wrapper)

the_tutor_client = tutor_client("messages.sqlite", generic_system_prompt, 0)

def entry_label(e, depth):
    if isinstance(e, type("")):
        return " " * depth + e
    else:
        return " " * depth + e[0]

def entry_children(e):
    if isinstance(e, type("")):
        return []
    else:
        return e[1]

class hierarchical_radios:
    def __init__(self):
        # topic option
        self.topic_other_box = widgets.Text(
            placeholder='Type topic here...',
            disabled=False,
            # layout=widgets.Layout(display='none')
        )
        self.topic_menu = topic_menu
        self.updating = 0
        chosen_path = (0,)
        topic_options = self.make_options(self.topic_menu, chosen_path, 0)
        self.paths = self.make_paths(self.topic_menu, chosen_path, ())
        self.topic_radios = widgets.RadioButtons(options=topic_options, disabled=False)
        self.topic_radios.observe(self.topic_chosen)
        # language option
        lang_options = ["Go", "Julia", "OCaml", "Rust"]
        self.lang_radios = widgets.RadioButtons(options=lang_options, disabled=False)
        # problem type option
        prob_type_options = ["introduction", "coding", "reading", "debugging"]
        self.prob_type_radios = widgets.RadioButtons(options=prob_type_options, disabled=False)
        # send button
        self.send_button = widgets.Button(description="Send")
        self.send_button.on_click(self.button_pressed)
        #
        self.output_area = widgets.Output()
    def make_paths(self, menu_tree, chosen_path, path):
        if DBG>=3:
            print(f"make_paths({menu_tree}, {chosen_path})")
        if chosen_path == ():
            path_list = [path + (i,) for i, e in enumerate(menu_tree)]
        else:
            p = chosen_path[0]
            path_list = ([path + (i,) for i, e in enumerate(menu_tree[:p+1])]
                         + self.make_paths(entry_children(menu_tree[p]), chosen_path[1:], path + (p,))
                         + [path + (p + 1 + i,) for i, e in enumerate(menu_tree[p+1:])])
        if DBG>=3:
            print(f"-> {path_list}")
        return path_list
    def make_options(self, menu_tree, chosen_path, depth):
        if DBG>=3:
            print(f"make_options({menu_tree}, {chosen_path})")
        if chosen_path == ():
            menu_list = [entry_label(e, depth) for e in menu_tree]
        else:
            p = chosen_path[0]
            menu_list = ([entry_label(e, depth) for e in menu_tree[:p+1]]
                         + self.make_options(entry_children(menu_tree[p]), chosen_path[1:], depth + 1)
                         + [entry_label(e, depth) for e in menu_tree[p+1:]])
        if DBG>=3:
            print(f"-> {menu_list}")
        return menu_list
    def topic_chosen(self, change):
        if self.updating:
            if DBG>=3:
                print(f"updating. return")
            return
        if DBG>=3:
            print(f"change = {change}")
        change_name = change["name"]
        if change_name == "index":
            idx = change["new"]
            lang = self.lang_radios.value
            if DBG>=3:
                print(f"chosen: {idx}, {lang}")
            chosen_path = self.paths[idx]
            if DBG>=3:
                print(f"chosen_path: {idx}")
            options = self.make_options(self.topic_menu, chosen_path, 0)
            self.paths = self.make_paths(self.topic_menu, chosen_path, ())
            self.updating = 1
            self.topic_radios.options = options
            self.topic_radios.index = idx
            self.updating = 0
    def make_prompt(self, prob_type, topic, lang):
        if prob_type == "introduction":
            return f'Explain the basics of "{topic}" of {lang} with an example or two'
        else:
            return f'Give me a {prob_type} problem on "{topic}" of {lang}'
        
    def button_pressed(self, b):
        topic = self.topic_radios.value
        if topic == "Other (describe below)":
            topic = self.topic_other_box.value
        lang = self.lang_radios.value
        prob_type = self.prob_type_radios.value
        prompt = self.make_prompt(prob_type, topic, lang)
        def res_wrapper(res):
            return f"### 💬 {prob_type} problem on {topic}:\n\n{res}"
        output_area = widgets.Output()
        with self.output_area:
            display(output_area)
        the_tutor_client.send_and_disp_in_area(prompt, res_wrapper, output_area)
        self.output_area = output_area
        with output_area:
            self.show()
        
    def show(self):
        display(self.topic_radios,
                self.topic_other_box,
                self.lang_radios,
                self.prob_type_radios,
                self.send_button)
        # self.output_area)


def Q(prompt):
    """
    General query
    AI に prompt を投げるだけ
    """
    def res_wrapper(res):
        return res
    return the_tutor_client.send_and_disp(prompt, res_wrapper)

def C(topic, plang):
    """
    Coding問題
    項目を言ってコーディング問題を生成してもらう
    """
    # prompt = f"「{topic}」に関するコーディングの問題を出して"
    prompt = f"Give me a coding problem on {topic} of {plang}"
    def res_wrapper(res):
        return f"### 💬 Coding problem on {topic}:\n\n{res}"
    return the_tutor_client.send_and_disp(prompt, res_wrapper)

def R(topic, plang):
    """
    Reading問題
    項目を言ってリーディング問題を生成してもらう
    """
    # prompt = f"「{topic}」に関するリーディングの問題を出して"
    prompt = f"Give me a reading problem on {topic} of {plang}"
    def res_wrapper(res):
        return f"### 💬 Reading problem on {topic}:\n\n{res}"
    return the_tutor_client.send_and_disp(prompt, res_wrapper)

def D(topic, plang):
    """
    Debug問題
    項目を言ってデバッグ問題を生成してもらう
    """
    # prompt = f"「{topic}」に関するデバッグの問題を出して"
    prompt = f"Give me a debugging problem on {topic} of {plang}"
    def res_wrapper(res):
        return f"### 💬 Debugging problem on {topic}:\n\n{res}"
    return the_tutor_client.send_and_disp(prompt, res_wrapper)

def I(topic, plang):
    """
    Introduce topic
    項目を言ってそれに関するイントロ
    """
    prompt = f"Explain {topic} of {plang}, including an example or two"
    def res_wrapper(res):
        return f"### 💬 Introduction to {topic}:\n\n{res}"
    return the_tutor_client.send_and_disp(prompt, res_wrapper)

# @register_cell_magic
def hey_tutor(line, cell):
    """
    %%call_tutor でそのセルのコードにフィードバックをもらう
    """
    def res_wrapper(res):
        return f"### 💬 Feedback:\n\n{res}"
    return the_tutor_client.send_and_disp(cell, res_wrapper)

# @register_cell_magic
def H(line, cell):
    """
    %%H でそのセルのコードにフィードバックをもらう
    (Pythonカーネルのみ使える方法)
    """
    def res_wrapper(res):
        return f"### 💬 Feedback:\n\n{res}"
    the_tutor_client.send_and_disp(cell, res_wrapper)

def T():
    """
    実行するとボタンとtext areaが現れ,
    text area にコードをコピーしてボタンを押すとフィードバックがもらえる
    (Pythonカーネルじゃなくても使える汎用的な方法)
    """
    input_area = widgets.Textarea(
        value='',
        placeholder='Paste your quesiton/answer here and press "Send"',
        layout=widgets.Layout(width='100%', height='200px')
    )
    output_area = widgets.Output()
    def res_wrapper(res):
        return f"### 💬 Response:\n\n{res}"
    def on_button_click(b):
        the_tutor_client.send_and_disp_in_area(input_area.value,
                                               res_wrapper,
                                               output_area)
    button = widgets.Button(description="💡 Send")
    button.on_click(on_button_click)
    display(widgets.VBox([input_area, button, output_area]))

def M():
    radios = hierarchical_radios()
    radios.show()
    
# x = C("""Recursive Functions""", "Julia")
# print(x)


In [ ]:
Q("再帰呼出しの問題を出して")

In [ ]:
T()